<a href="https://colab.research.google.com/github/seirah-yang/BERToping_visualization-/blob/main/preprocessing_validation(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# -*- coding: utf-8 -*-

import re
import os
import pandas as pd
import numpy as np
from datetime import datetime

BASE_DIR = "/content/drive/MyDrive/4_Network Analysis/nurse_voice"

date_str = datetime.today().strftime("%m%d")
output_dir = os.path.join(BASE_DIR, f"output{date_str}")

# 폴더가 없으면 자동 생성
os.makedirs(output_dir, exist_ok=True)

# 경로 설정
main_file_path = os.path.join(BASE_DIR, "maincorpus.txt")
supplement_file_path = os.path.join(BASE_DIR, "supplement.txt")

main_output_path = os.path.join(output_dir, "main_corpus_dedupli.csv")
main_excluded_path = os.path.join(output_dir, "main_doi_dupli_excluded.csv")

combined_output_path = os.path.join(output_dir, "final_result(main+supple).csv")
combined_excluded_path = os.path.join(output_dir, "final_combined_dupli_excluded(main+supple).csv")

supplement_output_dir = os.path.join(output_dir, "output_supplement")

In [4]:
#1. Publication Type 확인을 위한 정보 추가

doi_pattern = re.compile(
    r"(10\.[0-9]{4,}(?:\.[0-9]+)*/(?:[a-zA-Z0-9._;()/:+-]+))",
    re.IGNORECASE
)

def parse_pubmed_txt(file_path, corpus_name):
    records = []
    current = {}
    current_tag = None

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            tag_match = re.match(r"^([A-Z0-9]{2,4})\s*-\s(.*)$", line)

            if tag_match:
                tag = tag_match.group(1)
                value = tag_match.group(2).strip()
                current_tag = tag

                if tag == "PMID":
                    if current:
                        records.append(current)
                    current = {"Corpus": corpus_name}

                if tag in current:
                    if isinstance(current[tag], list):
                        current[tag].append(value)
                    else:
                        current[tag] = [current[tag], value]
                else:
                    current[tag] = value

            elif line.startswith("      ") and current_tag:
                cont = line.strip()
                if current_tag in current:
                    if isinstance(current[current_tag], list):
                        current[current_tag][-1] += " " + cont
                    else:
                        current[current_tag] += " " + cont

    if current:
        records.append(current)

    df = pd.DataFrame(records)

    # 약어 정리 dictionary
    df = df.rename(columns={
        "PMID": "PMID",
        "TI": "Title",
        "AB": "Abstract",
        "DP": "Date",
        "JT": "Journal",
        "PT": "PublicationType",
        "MH": "MeSHTerms",
        "OT": "Keywords",
        "AU": "Authors",
        "AID": "AID",
        "LID": "LID"
    })


    for col in df.columns:
        df[col] = df[col].apply(
            lambda x: "; ".join(x) if isinstance(x, list) else x
        )

    # DOI, Year, PMID, Title, Abstract, Journal, PublicationType columns 추출
    def extract_doi(row):
        for col in ["AID", "LID"]:
            val = str(row.get(col, ""))
            match = doi_pattern.search(val)
            if match:
                return match.group(1).rstrip(".")
        return None

    df["DOI"] = df.apply(extract_doi, axis=1)


    if "Date" in df.columns:
        df["Year"] = df["Date"].astype(str).str.extract(r"(\d{4})")
    else:
        df["Year"] = None


    for col in ["PMID", "Title", "Abstract", "Journal", "PublicationType", "DOI", "Year"]:
        if col not in df.columns:
            df[col] = None

    df = df.dropna(subset=["PMID"]).reset_index(drop=True)

    return df

In [5]:
def deduplicate_pubmed_records(df, corpus_label):

    print(f"\n {corpus_label}")

    initial_count = len(df)
    print(f"초기 입력 문헌 수: {initial_count}")

    # 1) PMID 기준 중복 제거
    df_pmid_dedup = df.drop_duplicates(subset="PMID", keep="first").copy()
    pmid_dedup_count = len(df_pmid_dedup)

    print(f"PMID 중복 제거 후 문헌 수: {pmid_dedup_count}")
    print(f"PMID 제외된 문헌 수: {initial_count - pmid_dedup_count}")

    # 2) DOI 결측 분리
    df_with_doi = df_pmid_dedup[df_pmid_dedup["DOI"].notna()].copy()
    df_without_doi = df_pmid_dedup[df_pmid_dedup["DOI"].isna()].copy()

    print(f"DOI 있는 문헌 수: {len(df_with_doi)}")
    print(f"DOI 없는 문헌 수: {len(df_without_doi)}")

    # 3) DOI가 있는 문헌 중복 제거 & DOI 중복 제외 목록
    df_with_doi_dedup = df_with_doi.drop_duplicates(subset="DOI", keep="first").copy()

    excluded_doi_duplicates = df_with_doi[
        df_with_doi.duplicated(subset="DOI", keep="first")
    ].copy()

    print(f"DOI가 있으면서, 중복 제거 후 : {len(df_with_doi_dedup)}")
    print(f"중복으로 제거된 문헌 수: {len(excluded_doi_duplicates)}")

    # 4) DOI 없는 문헌은 보존
    df_final = pd.concat(
        [df_with_doi_dedup, df_without_doi],
        ignore_index=True
    )

    df_final = df_final.sort_values(by="PMID").reset_index(drop=True)

    print(f"최종 중복된 문헌 제외 문헌 수: {len(df_final)}")

    return df_final, excluded_doi_duplicates

# Main Corpus(메인 축: Speaking Up + Voice Behavior) 파싱

main_df = parse_pubmed_txt(main_file_path, corpus_name="Main")
main_dedup, main_excluded_doi = deduplicate_pubmed_records(
    main_df,
    corpus_label="Main Corpus: Speaking Up + Voice Behavior"
)

main_dedup.to_csv(main_output_path, index=False, encoding="utf-8-sig")
main_excluded_doi.to_csv(main_excluded_path, index=False, encoding="utf-8-sig")

print(f"\nMain corpus saved to: {main_output_path}")
print(f"Main DOI duplicate excluded list saved to: {main_excluded_path}")



 Main Corpus: Speaking Up + Voice Behavior
초기 입력 문헌 수: 2980
PMID 중복 제거 후 문헌 수: 2980
PMID 제외된 문헌 수: 0
DOI 있는 문헌 수: 2585
DOI 없는 문헌 수: 395
DOI가 있으면서, 중복 제거 후 : 2585
중복으로 제거된 문헌 수: 0
최종 중복된 문헌 제외 문헌 수: 2980

Main corpus saved to: /content/drive/MyDrive/4_Network Analysis/nurse_voice/output0612/main_corpus_dedupli.csv
Main DOI duplicate excluded list saved to: /content/drive/MyDrive/4_Network Analysis/nurse_voice/output0612/main_doi_dupli_excluded.csv


In [6]:
# 4. Supplementary Corpus(보조축: Assertive Communication) parsing
os.makedirs(supplement_output_dir, exist_ok=True)

supplement_dedup_output_path = os.path.join(supplement_output_dir, "supplement_deduplicated_0603.csv")
supplement_excluded_path = os.path.join(supplement_output_dir, "supplement_doi_duplicate_excluded_0603.csv")

supplement_df = parse_pubmed_txt(supplement_file_path, corpus_name="Supplement")
supplement_dedup, supplement_excluded_doi = deduplicate_pubmed_records(
    supplement_df,
    corpus_label="Supplementary Corpus: Assertive Communication"
)

supplement_dedup.to_csv(supplement_dedup_output_path, index=False, encoding="utf-8-sig")
supplement_excluded_doi.to_csv(supplement_excluded_path, index=False, encoding="utf-8-sig")

print(f"\nSupplement corpus saved to: {supplement_dedup_output_path}")
print(f"Supplement DOI duplicate excluded list saved to: {supplement_excluded_path}")

# 5. Main + Supplement 결합

combined_df = pd.concat(
    [main_dedup, supplement_dedup],
    ignore_index=True
)

print("\n Main Corpus + Supplement Corpus")
print(f"Main Corpus + Supplement Corpus 초기 입력 문헌 수: {len(combined_df)}")

# 1) 결합 후 PMID 기준 중복 제거
combined_pmid_dedup = combined_df.drop_duplicates(subset="PMID", keep="first").copy()

print(f"결합 후 PMID 기준 중복 제거 후 결과: {len(combined_pmid_dedup)}")
print(f"결합 후 PMID 중복 제외 결과: {len(combined_df) - len(combined_pmid_dedup)}")

# 2) DOI가 있는 문헌만 DOI 중복 제거
combined_with_doi = combined_pmid_dedup[combined_pmid_dedup["DOI"].notna()].copy()
combined_without_doi = combined_pmid_dedup[combined_pmid_dedup["DOI"].isna()].copy()

combined_with_doi_dedup = combined_with_doi.drop_duplicates(subset="DOI", keep="first").copy()

combined_excluded_doi = combined_with_doi[
    combined_with_doi.duplicated(subset="DOI", keep="first")
].copy()

# 3) DOI 없는 문헌 보존
final_combined_df = pd.concat(
    [combined_with_doi_dedup, combined_without_doi],
    ignore_index=True
)

final_combined_df = final_combined_df.sort_values(by="PMID").reset_index(drop=True)

print(f"DOI(+) & PMID 중복 제거된 문헌: {len(combined_with_doi)}")
print(f"DOI(-) & PMID 중복 제거된 문헌: {len(combined_without_doi)}")
print(f"DOI 중복 제거: {len(combined_excluded_doi)}")
print(f"Final combined 문헌 수: {len(final_combined_df)}")


 Supplementary Corpus: Assertive Communication
초기 입력 문헌 수: 262
PMID 중복 제거 후 문헌 수: 262
PMID 제외된 문헌 수: 0
DOI 있는 문헌 수: 198
DOI 없는 문헌 수: 64
DOI가 있으면서, 중복 제거 후 : 198
중복으로 제거된 문헌 수: 0
최종 중복된 문헌 제외 문헌 수: 262

Supplement corpus saved to: /content/drive/MyDrive/4_Network Analysis/nurse_voice/output0612/output_supplement/supplement_deduplicated_0603.csv
Supplement DOI duplicate excluded list saved to: /content/drive/MyDrive/4_Network Analysis/nurse_voice/output0612/output_supplement/supplement_doi_duplicate_excluded_0603.csv

 Main Corpus + Supplement Corpus
Main Corpus + Supplement Corpus 초기 입력 문헌 수: 3242
결합 후 PMID 기준 중복 제거 후 결과: 3208
결합 후 PMID 중복 제외 결과: 34
DOI(+) & PMID 중복 제거된 문헌: 2750
DOI(-) & PMID 중복 제거된 문헌: 458
DOI 중복 제거: 0
Final combined 문헌 수: 3208


In [7]:
# 6. Missing DOI 확인

missing_doi_final = final_combined_df[final_combined_df["DOI"].isna()]

print("\n [Missing DOI Check]")
print(f"Final records without DOI: {len(missing_doi_final)}")

if len(missing_doi_final) > 0:
    display(missing_doi_final[["Corpus", "PMID", "Title"]].head(20))

# 7. final result 저장

final_combined_df.to_csv(combined_output_path, index=False, encoding="utf-8-sig")
combined_excluded_doi.to_csv(combined_excluded_path, index=False, encoding="utf-8-sig")

print(f"\nFinal combined result saved to: {combined_output_path}")
print(f"Final combined DOI duplicate excluded list saved to: {combined_excluded_path}")

summary = pd.DataFrame({
    "Step": [
        "Main initial",
        "Main final deduplicated",
        "Supplement initial",
        "Supplement final deduplicated",
        "Combined before final deduplication",
        "Final combined deduplicated",
        "Final records without DOI preserved"
    ],
    "Count": [
        len(main_df),
        len(main_dedup),
        len(supplement_df),
        len(supplement_dedup),
        len(combined_df),
        len(final_combined_df),
        len(missing_doi_final)
    ]
})

display(summary)


 [Missing DOI Check]
Final records without DOI: 458


,Corpus,PMID,Title
0,Main,10063504,[New research and considerations in managing m...
6,Main,10141931,Employees speak out. Testimonials help hospita...
7,Main,10147312,[{Importance and realization of expectations o...
8,Main,10160949,Ethical issues faced by clinician/managers in ...
12,Main,10205545,Higher level practice: the consumer's perspect...
15,Main,10223056,Ethics on the job: a survey. The realities of ...
17,Main,10280879,Movers and shakers.
18,Supplement,10288872,Confrontation: an underused nursing management...
19,Main,10290550,Directors of nursing speak out on LTC patients...
20,Main,10292846,Speaking up for nurses. Concerted political ac...



Final combined result saved to: /content/drive/MyDrive/4_Network Analysis/nurse_voice/output0612/final_result(main+supple).csv
Final combined DOI duplicate excluded list saved to: /content/drive/MyDrive/4_Network Analysis/nurse_voice/output0612/final_combined_dupli_excluded(main+supple).csv


,Step,Count
0,Main initial,2980
1,Main final deduplicated,2980
2,Supplement initial,262
3,Supplement final deduplicated,262
4,Combined before final deduplication,3242
5,Final combined deduplicated,3208
6,Final records without DOI preserved,458


In [8]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.2 MB/s eta 0:00:00


In [9]:
from Bio import Entrez # pip install biopython
from tqdm import tqdm

Entrez.email="oktjdus@yonsei.ac.kr"

def get_retracted_pmids(pmids, batch_size=200):

    retracted_pmids = []

    for i in tqdm(range(0, len(pmids), batch_size)):
        batch = pmids[i:i+batch_size]

        handle = Entrez.efetch(
            db="pubmed",
            id=",".join(batch),
            rettype="medline",
            retmode="text"
        )

        text = handle.read()
        handle.close()

        records = text.split("\nPMID- ")

        for rec in records:

            if "Retracted Publication" in rec:
                pmid_match = rec.split("\n")[0].strip()

                if pmid_match.isdigit():
                    retracted_pmids.append(pmid_match)
    return retracted_pmids

In [10]:
pmid_list = final_combined_df["PMID"].astype(str).tolist()

retracted_pmids = get_retracted_pmids(pmid_list)

print(f"Retracted papers found: {len(retracted_pmids)}")

# 철회문헌 제거
retracted_df = final_combined_df[
    final_combined_df["PMID"].isin(retracted_pmids)
].copy()

final_clean_df = final_combined_df[
    ~final_combined_df["PMID"].isin(retracted_pmids)
].copy()

print(f"Before removal: {len(final_combined_df)}")
print(f"Retracted removed: {len(retracted_df)}")
print(f"Final corpus: {len(final_clean_df)}")

100%|██████████| 17/17 [00:22<00:00,  1.35s/it]

Retracted papers found: 1
Before removal: 3208
Retracted removed: 1
Final corpus: 3207


In [11]:
# 철회문헌 제거 후 결과 저장
retracted_df.to_csv(
    "/content/drive/MyDrive/4_Network Analysis/nurse_voice/output0608/retracted_df.csv", # 여기 경로는 직접기입함
    index=False,
    encoding="utf-8-sig"
)

final_clean_df.to_csv(
    "/content/drive/MyDrive/4_Network Analysis/nurse_voice/output0608/final_analytic_corpus.csv", # 여기 경로는 직접기입함
    index=False,
    encoding="utf-8-sig"
)

print(f"저장완료: retracted_df ({len(retracted_df)}건)")
print(f"저장완료: final_clean_df ({len(final_clean_df)}건)")

# Records removed before screening: Retracted publications (n = 1) -> 논문에 추후 기입 또는 Figma 그릴때 필요

print(final_combined_df.columns.tolist())

저장완료: retracted_df (1건)
저장완료: final_clean_df (3207건)
['Corpus', 'PMID', 'OWN', 'STAT', 'DCOM', 'LR', 'IS', 'VI', 'IP', 'Date', 'Title', 'PG', 'LID', 'Abstract', 'FAU', 'Authors', 'AD', 'LA', 'PublicationType', 'PL', 'TA', 'Journal', 'JID', 'MeSHTerms', 'RF', 'EDAT', 'MHDA', 'CRDT', 'PHST', 'AID', 'PST', 'SO', 'SB', 'OTO', 'Keywords', 'CI', 'DEP', 'COIS', 'AUID', 'GR', 'PMC', 'PMCR', 'CN', 'CIN', 'OID', 'GN', 'RN', 'EIN', 'MID', 'TT', 'OAB', 'OABL', 'SI', 'CON', 'EFR', 'UOF', 'CTDT', 'PB', 'BTI', 'RPI', 'FIR', 'IR', 'UIN', 'DRDT', 'FED', 'ED', 'PS', 'FPS', 'RIN', 'ISBN', 'CP', 'RPF', 'CTI', 'DOI', 'Year', 'CRI', 'CRF']


In [12]:
# 1. 제거되어야 할 문헌 정의

exclude_types = [
    "Retracted Publication",
    "Retraction of Publication",
    "Published Erratum",
    "Editorial",
    "Comment",
    "News",
    "Interview",
    "Biography"
]

pattern = "|".join(exclude_types)


excluded_pubtype_df = final_combined_df[
    final_combined_df["PublicationType"].fillna("").str.contains(
        pattern,
        case=False,
        regex=True
    )
].copy()

final_analytic_df = final_combined_df[
    ~final_combined_df["PublicationType"].fillna("").str.contains(
        pattern,
        case=False,
        regex=True
    )
].copy()

print(f"Before filtering : {len(final_combined_df):,}")
print(f"Excluded records : {len(excluded_pubtype_df):,}")
print(f"Final corpus     : {len(final_analytic_df):,}")


display(
    excluded_pubtype_df[
        ["PMID", "Title", "PublicationType"]
    ].head(20)
)

Before filtering : 3,208
Excluded records : 97
Final corpus     : 3,111


,PMID,Title,PublicationType
128,11894455,Nurses speak out about microprem babies.,News
131,11908123,Nurses speaking out about what they do. Interv...,Interview
135,11962992,"Whistleblowing. My message: if in doubt, speak...",Interview
201,12581397,Modernizing the British National Health Servic...,Comment; Journal Article; Review
212,12735076,Client narratives: a theoretical perspective.,Comment; Journal Article; Review
216,12776636,What role should education standards play in t...,Editorial; Review
254,1471807,Nurses and the nation speak up for change.,Editorial
262,1491847,Nurses must speak up.,Editorial
263,14963949,Speaking up for children. Interview by Cathari...,Interview
303,15356905,Speaking out for children. Interview by Janis ...,Interview


In [13]:
# 2. Letter / Comment 잔여 여부 확인
# 위에서 제거 하고도 Letter or Comment 문헌이 남아 있는지 확인 후, 남아 있다면 제거

letter_comment_terms = [
    "Letter",
    "Comment"
]

letter_comment_pattern = "|".join(letter_comment_terms)

letter_comment_df = final_analytic_df[
    final_analytic_df["PublicationType"].fillna("").str.contains(
        letter_comment_pattern,
        case=False,
        regex=True
    )
].copy()

print(f"Current corpus: {len(final_analytic_df):,}")
print(f"Letter/Comment candidates: {len(letter_comment_df):,}")


if len(letter_comment_df) > 0:
    display(
        letter_comment_df[
            ["PMID", "Title", "Year", "Journal", "PublicationType"]
        ].head(100)
    )

"""### 3. 최종 데이터셋에서 Letter / Comment 유형 추가 제거"""

current_final_analytic_df_len = len(final_analytic_df)

# Exclude 'Letter' and 'Comment' from the final_analytic_df
final_analytic_df = final_analytic_df[
    ~final_analytic_df["PMID"].isin(letter_comment_df["PMID"])
].copy()

# Add the newly excluded 'Letter' and 'Comment' documents to the excluded_pubtype_df
excluded_pubtype_df = pd.concat([excluded_pubtype_df, letter_comment_df], ignore_index=True)

print(f"Before additional filtering: {current_final_analytic_df_len:,}")
print(f"Removed 'Letter'/'Comment' documents: {len(letter_comment_df):,}")
print(f"Final corpus after additional filtering: {len(final_analytic_df):,}")

Current corpus: 3,111
Letter/Comment candidates: 21


,PMID,Title,Year,Journal,PublicationType
245,14657735,NPs speak out about the less fortunate.,2003,The Nurse practitioner,Letter
246,14671745,NPs speak out about the less fortunate.,2003,The Nurse practitioner,Letter
281,15156857,Survey raises concerns.,2004,British journal of nursing (Mark Allen Publish...,Letter
319,15600012,Readers speak out: are nursing schools really ...,2004,RN,Letter
320,15600013,Readers speak out: are nursing schools really ...,2004,RN,Letter
321,15600014,Readers speak out: are nursing schools really ...,2004,RN,Letter
411,16579148,Nurses speak out on voting for their manager.,2006,RN,Letter
412,16579149,Nurses speak out on voting for their manager.,2006,RN,Letter
413,16579150,Nurses speak out on voting for their manager.,2006,RN,Letter
414,16579151,Nurses speak out on voting for their manager.,2006,RN,Letter


Before additional filtering: 3,111
Removed 'Letter'/'Comment' documents: 21
Final corpus after additional filtering: 3,090


In [14]:
print(final_combined_df.columns.tolist())
# [중요!]'PublicationType'이 없으면 위에 파서하는 과정 코드 수정 필요

['Corpus', 'PMID', 'OWN', 'STAT', 'DCOM', 'LR', 'IS', 'VI', 'IP', 'Date', 'Title', 'PG', 'LID', 'Abstract', 'FAU', 'Authors', 'AD', 'LA', 'PublicationType', 'PL', 'TA', 'Journal', 'JID', 'MeSHTerms', 'RF', 'EDAT', 'MHDA', 'CRDT', 'PHST', 'AID', 'PST', 'SO', 'SB', 'OTO', 'Keywords', 'CI', 'DEP', 'COIS', 'AUID', 'GR', 'PMC', 'PMCR', 'CN', 'CIN', 'OID', 'GN', 'RN', 'EIN', 'MID', 'TT', 'OAB', 'OABL', 'SI', 'CON', 'EFR', 'UOF', 'CTDT', 'PB', 'BTI', 'RPI', 'FIR', 'IR', 'UIN', 'DRDT', 'FED', 'ED', 'PS', 'FPS', 'RIN', 'ISBN', 'CP', 'RPF', 'CTI', 'DOI', 'Year', 'CRI', 'CRF']


In [15]:
# 제거된 21개 문헌 확인
print("\n--- Excluded Letter/Comment Documents (last 21 rows) ---")
display(excluded_pubtype_df.tail(21))

# 최종 데이터셋 저장
final_analytic_output_path = os.path.join(BASE_DIR, "output0608/final_analytic_corpus_cleaned.csv")
final_analytic_df.to_csv(final_analytic_output_path, index=False, encoding="utf-8-sig")
print(len(excluded_pubtype_df))
print(f"\nFinal analytic corpus saved to: {final_analytic_output_path}")


--- Excluded Letter/Comment Documents (last 21 rows) ---


,Corpus,PMID,OWN,STAT,DCOM,LR,IS,VI,IP,Date,...,FPS,RIN,ISBN,CP,RPF,CTI,DOI,Year,CRI,CRF
97,Main,14657735,NLM,MEDLINE,20040204,20191026,0361-1817 (Print); 0361-1817 (Linking),28,11,2003 Nov,...,NaN,NaN,NaN,NaN,NaN,NaN,10.1097/00006205-200311000-00002,2003,NaN,NaN
98,Main,14671745,NLM,MEDLINE,20040204,20191026,0361-1817 (Print); 0361-1817 (Linking),28,11,2003 Nov,...,NaN,NaN,NaN,NaN,NaN,NaN,10.1097/00006205-200311000-00004,2003,NaN,NaN
99,Main,15156857,NLM,MEDLINE,20040702,20260128,0966-0461 (Print); 0966-0461 (Linking),13,8,2004 Apr 22-May 12,...,NaN,NaN,NaN,NaN,NaN,NaN,None,2004,NaN,NaN
100,Main,15600012,NLM,MEDLINE,20050202,20041216,0033-7021 (Print); 0033-7021 (Linking),67,11,2004 Nov,...,NaN,NaN,NaN,NaN,NaN,NaN,None,2004,NaN,NaN
101,Main,15600013,NLM,MEDLINE,20050202,20041216,0033-7021 (Print); 0033-7021 (Linking),67,11,2004 Nov,...,NaN,NaN,NaN,NaN,NaN,NaN,None,2004,NaN,NaN
102,Main,15600014,NLM,MEDLINE,20050202,20160803,0033-7021 (Print); 0033-7021 (Linking),67,11,2004 Nov,...,NaN,NaN,NaN,NaN,NaN,NaN,None,2004,NaN,NaN
103,Main,16579148,NLM,MEDLINE,20060518,20060403,0033-7021 (Print); 0033-7021 (Linking),69,3,2006 Mar,...,NaN,NaN,NaN,NaN,NaN,NaN,None,2006,NaN,NaN
104,Main,16579149,NLM,MEDLINE,20060518,20060403,0033-7021 (Print); 0033-7021 (Linking),69,3,2006 Mar,...,NaN,NaN,NaN,NaN,NaN,NaN,None,2006,NaN,NaN
105,Main,16579150,NLM,MEDLINE,20060518,20060403,0033-7021 (Print); 0033-7021 (Linking),69,3,2006 Mar,...,NaN,NaN,NaN,NaN,NaN,NaN,None,2006,NaN,NaN
106,Main,16579151,NLM,MEDLINE,20060518,20060403,0033-7021 (Print); 0033-7021 (Linking),69,3,2006 Mar,...,NaN,NaN,NaN,NaN,NaN,NaN,None,2006,NaN,NaN


118

Final analytic corpus saved to: /content/drive/MyDrive/4_Network Analysis/nurse_voice/output0608/final_analytic_corpus_cleaned.csv


## PCC 자동 라벨링 및 수동 검증 구조 재정렬
#
#### Cell 1  PCC Rule 생성
#### Cell 2  PCC_auto_label 생성
#### Cell 3  Needs review export
#### Cell 4  Validation file load
#### Cell 5  manual_binary 생성
#### Cell 6  PMID merge
#### Cell 7  crosstab
#### Cell 8  confusion matrix
#### Cell 9  Needs review 분석
####Cell 10 Noise 분석

In [16]:
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score

# 최종 analytic corpus 불러오기
df = pd.read_csv(final_analytic_output_path)

TITLE_COL = "Title"
ABSTRACT_COL = "Abstract"
KEYWORD_COL = "Keywords"
MESH_COL = "MeSHTerms"

# screening에 사용할 text 구성
# Keywords/MeSHTerms가 있으면 같이 넣어 sensitivity를 높임
text_cols_for_screening = [TITLE_COL, ABSTRACT_COL]
for optional_col in [KEYWORD_COL, MESH_COL, "PublicationType", "Journal"]:
    if optional_col in df.columns:
        text_cols_for_screening.append(optional_col)

for col in text_cols_for_screening:
    if col not in df.columns:
        df[col] = ""

# 결측값 방어
df["text_for_screening"] = (
    df[text_cols_for_screening]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
)

print("Text columns used for screening:", text_cols_for_screening)
display(df[["PMID", "Title", "text_for_screening"]].head())

# Population: nurse/nursing 관련
p_patterns = [
    r"\bnurs\w*\b",
    r"\bregistered nurse\w*\b",
    r"\brn\b",
    r"\bnursing staff\b",
]

Text columns used for screening: ['Title', 'Abstract', 'Keywords', 'MeSHTerms', 'PublicationType', 'Journal']


,PMID,Title,text_for_screening
0,10063504,[New research and considerations in managing m...,[new research and considerations in managing m...
1,10085849,Nurse-physician collaboration.,nurse-physician collaboration. the literature ...
2,10111764,Certificate of need and nursing home bed capac...,certificate of need and nursing home bed capac...
3,10115034,Mental illness and psychotropic medication use...,mental illness and psychotropic medication use...
4,10137905,Continuity of care in maternity services--the ...,continuity of care in maternity services--the ...


In [17]:
noise_patterns = [
    r"\bvoice disorder\w*\b",
    r"\bvocal cord\w*\b",
    r"\blaryngeal\b",
    r"\bspeech therap\w*\b",
    r"\bspeech pathology\b",
    r"\bvoice recognition\b",
    r"\bvoice assistant\b",
    r"\bacoustic analysis\b",
]

# Exclusion audit: silence-only 연구 확인용
# 주의: 자동 제외에는 직접 사용하지 않고, Needs review/False positive 점검에 사용
silence_patterns = [
    r"\borganizational silence\b",
    r"\borganisational silence\b",
    r"\bemployee silence\b",
    r"\bsilence behavior\b",
    r"\bsilence behaviour\b",
]

def contains_any(text, patterns):
    if pd.isna(text):
        text = ""
    return any(re.search(p, str(text), flags=re.IGNORECASE) for p in patterns)


### 조건
PCC ≥ 2 & Noise = False		Likely relevant

PCC ≥ 2 & Noise = True		Likely relevant

PCC = 1                     Needs review

PCC = 0	                	Likely irrelevant

In [19]:
p_patterns = [
    r"\bnurs\w*\b",
    r"\bregistered nurse\w*\b",
    r"\brn\b",
    r"\bnursing staff\b",
]
c_patterns = [
    r"\bspeak(?:ing)? up\b",
    r"\bspeak(?:ing)? out\b",
    r"\brais\w* concern\w*\b",
    r"\bvoice behavior\b",
    r"\bvoice behaviour\b",
    r"\bemployee voice\b",
    r"\bsafety voice\b",
    r"\bpromotive voice\b",
    r"\bprohibitive voice\b",
    r"\bassertive\w* communicat\w*\b",
    r"\bassertiveness\b",
    r"\bwhistleblow\w*\b",
]

# Context: healthcare setting
ctx_patterns = [
    r"\bhealthcare\b",
    r"\bhealth care\b",
    r"\bhospital\w*\b",
    r"\bclinical\b",
    r"\bpatient safety\b",
    r"\bemergency department\b",
    r"\boutpatient\b",
    r"\blong-term care\b",
    r"\bnursing home\b",
    r"\bward\b",
    r"\bunit\b",
]

In [20]:
df["P_nurse_auto"] = df["text_for_screening"].apply(lambda x: contains_any(x, p_patterns))
df["C_speakup_voice_auto"] = df["text_for_screening"].apply(lambda x: contains_any(x, c_patterns))
df["Ctx_healthcare_auto"] = df["text_for_screening"].apply(lambda x: contains_any(x, ctx_patterns))
df["Noise_auto"] = df["text_for_screening"].apply(lambda x: contains_any(x, noise_patterns))
df["Silence_auto"] = df["text_for_screening"].apply(lambda x: contains_any(x, silence_patterns))

# PCC score = Population + Concept + Context
df["PCC_score"] = (
    df["P_nurse_auto"].astype(int)
    + df["C_speakup_voice_auto"].astype(int)
    + df["Ctx_healthcare_auto"].astype(int)
)


df["PCC_auto_label"] = np.select(
    [
        df["PCC_score"] >= 2,
        df["PCC_score"] == 1,
        df["PCC_score"] == 0,
    ],
    [
        "Likely relevant",
        "Needs review",
        "Likely irrelevant",
    ],
    default="Needs review"
)

pcc_auto_output_path = os.path.join(output_dir, "pcc_auto_mapped_corpus.csv")
df.to_csv(pcc_auto_output_path, index=False, encoding="utf-8-sig")

print(f"PCC auto-mapped corpus saved to: {pcc_auto_output_path}")
display(df["PCC_auto_label"].value_counts())
display(pd.crosstab(df["PCC_score"], df["PCC_auto_label"], margins=True))

PCC auto-mapped corpus saved to: /content/drive/MyDrive/4_Network Analysis/nurse_voice/output0612/pcc_auto_mapped_corpus.csv


,count
PCC_auto_label,
Likely relevant,2668
Needs review,418
Likely irrelevant,4


PCC_auto_label,Likely irrelevant,Likely relevant,Needs review,All
PCC_score,,,,
0,4,0,0,4
1,0,0,418,418
2,0,1604,0,1604
3,0,1064,0,1064
All,4,2668,418,3090


### 'Needs Review' 문서 추출

In [21]:
needs_review_df = df[df["PCC_auto_label"] == "Needs review"].copy()
needs_review_output_path = os.path.join(output_dir, "pcc_needs_review_corpus.csv")
needs_review_df.to_csv(needs_review_output_path, index=False, encoding="utf-8-sig")

print(f"'Needs review' 문서 {len(needs_review_df)}개가 저장되었습니다: {needs_review_output_path}")
display(needs_review_df[["PMID", "Title", "Abstract", "PCC_score", "P_nurse_auto", "C_speakup_voice_auto", "Ctx_healthcare_auto", "Noise_auto"]].head())


# Cell 4. Validation file load


validation_file_path = os.path.join(BASE_DIR, "pcc_validation_sample_for_manual_review_completed.xlsx")

if os.path.exists(validation_file_path):
    val = pd.read_excel(validation_file_path)
    print("Validation file loaded:", validation_file_path)
    print("Available columns in val DataFrame:", val.columns.tolist())
else:
    val = None
    print("Validation file not found. Manual validation steps skipped:", validation_file_path)



'Needs review' 문서 418개가 저장되었습니다: /content/drive/MyDrive/4_Network Analysis/nurse_voice/output0612/pcc_needs_review_corpus.csv


,PMID,Title,Abstract,PCC_score,P_nurse_auto,C_speakup_voice_auto,Ctx_healthcare_auto,Noise_auto
4,10137905,Continuity of care in maternity services--the ...,Examines issues raised by Changing Childbirth....,1,True,False,False,False
12,10205545,Higher level practice: the consumer's perspect...,New nursing job titles mean little to the gene...,1,True,False,False,False
16,10226725,Caries-preventive methods used for children an...,"Denmark, Iceland, Norway, and Sweden have all ...",1,True,False,False,False
32,10455616,Ovarian cancer: an update for nursing practice.,A diagnosis of ovarian cancer is a crisis for ...,1,True,False,False,False
34,10457230,Considering the nature of intersubjectivity wi...,The notion of intersubjectivity raises fundame...,1,True,False,False,False


Validation file loaded: /content/drive/MyDrive/4_Network Analysis/nurse_voice/pcc_validation_sample_for_manual_review_completed.xlsx
Available columns in val DataFrame: ['Corpus', 'PMID', 'OWN', 'STAT', 'DCOM', 'LR', 'IS', 'VI', 'IP', 'Date', 'Title', 'PG', 'LID', 'Abstract', 'FAU', 'Authors', 'AD', 'LA', 'PublicationType', 'PL', 'TA', 'Journal', 'JID', 'MeSHTerms', 'RF', 'EDAT', 'MHDA', 'CRDT', 'PHST', 'AID', 'PST', 'SO', 'SB', 'OTO', 'Keywords', 'CI', 'DEP', 'COIS', 'AUID', 'GR', 'PMC', 'PMCR', 'CN', 'CIN', 'OID', 'GN', 'RN', 'EIN', 'MID', 'TT', 'OAB', 'OABL', 'SI', 'CON', 'EFR', 'UOF', 'CTDT', 'PB', 'BTI', 'RPI', 'FIR', 'IR', 'UIN', 'DRDT', 'FED', 'ED', 'PS', 'FPS', 'RIN', 'ISBN', 'CP', 'RPF', 'CTI', 'DOI', 'Year', 'CRI', 'CRF', 'text_for_screening', 'P_nurse', 'C_speakup_voice', 'Ctx_healthcare', 'final_relevance', 'reviewer_note', 'Relevance_Reevaluated']


In [22]:


# Cell 5. manual_binary 생성


if val is not None:
    # 수동 검토 결과 컬럼명 우선순위
    manual_label_candidates = [
        "final_relevance",
        "manual_label",
        "manual_relevance",
        "relevance",
        "label",
    ]

    manual_label_col = None
    for col in manual_label_candidates:
        if col in val.columns:
            manual_label_col = col
            break

    if manual_label_col is None:
        raise ValueError(
            "Manual label column not found. Expected one of: "
            + ", ".join(manual_label_candidates)
        )

    print("Manual label column used:", manual_label_col)
    display(val[manual_label_col].value_counts(dropna=False))

    label_dict = {
        "relevant": "relevant",
        "include": "relevant",
        "included": "relevant",
        "1": "relevant",
        "yes": "relevant",
        "y": "relevant",
        "포함": "relevant",

        "irrelevant": "irrelevant",
        "exclude": "irrelevant",
        "excluded": "irrelevant",
        "0": "irrelevant",
        "no": "irrelevant",
        "n": "irrelevant",
        "불포함": "irrelevant",
    }

    val["manual_label_clean"] = (
        val[manual_label_col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    val["manual_binary"] = val["manual_label_clean"].map(label_dict)

    unmapped = val[val["manual_binary"].isna()][manual_label_col].value_counts(dropna=False)
    if len(unmapped) > 0:
        print("Warning: Unmapped manual labels found. Check these values:")
        display(unmapped)

    # manual label이 정상적으로 매핑된 행만 평가 대상으로 사용
    val_eval = val[val["manual_binary"].notna()].copy()
    print(f"Manual validation rows usable for evaluation: {len(val_eval)} / {len(val)}")


Manual label column used: final_relevance


,count
final_relevance,
NaN,300


,count
final_relevance,
NaN,300


Manual validation rows usable for evaluation: 0 / 300


In [23]:
# Cell 6. PMID merge
if val is not None:
    if "PMID" not in df.columns or "PMID" not in val_eval.columns:
        raise ValueError("PMID column not found in df or validation file.")

    # merge 안정성을 위해 PMID를 문자열로 통일
    df_merge = df[[
        "PMID",
        "PCC_auto_label",
        "PCC_score",
        "P_nurse_auto",
        "C_speakup_voice_auto",
        "Ctx_healthcare_auto",
        "Noise_auto",
        "Silence_auto",
    ]].copy()

    df_merge["PMID"] = df_merge["PMID"].astype(str).str.strip()
    val_eval["PMID"] = val_eval["PMID"].astype(str).str.strip()

    # validation 파일에 기존 자동 라벨 컬럼이 있으면 중복 방지
    cols_to_drop = [
        "PCC_auto_label",
        "PCC_score",
        "P_nurse_auto",
        "C_speakup_voice_auto",
        "Ctx_healthcare_auto",
        "Noise_auto",
        "Silence_auto",
    ]
    val_eval = val_eval.drop(columns=[c for c in cols_to_drop if c in val_eval.columns], errors="ignore")

    val_eval = pd.merge(val_eval, df_merge, on="PMID", how="left")

    unmatched_n = val_eval["PCC_auto_label"].isna().sum()
    if unmatched_n > 0:
        print(f"Warning: {unmatched_n} PMIDs in validation file could not be matched with PCC_auto_label from df.")
        display(val_eval[val_eval["PCC_auto_label"].isna()][["PMID", manual_label_col]].head(20))

    val_eval = val_eval[val_eval["PCC_auto_label"].notna()].copy()
    print(f"Validation rows matched to auto labels: {len(val_eval)}")

Validation rows matched to auto labels: 0


In [25]:
# Cell 7. crosstab
if val is not None:
    print("\n--- Manual binary vs PCC_auto_label ---")
    ct = pd.crosstab(
        val_eval["manual_binary"],
        val_eval["PCC_auto_label"],
        margins=True
    )
    display(ct)

    print("\n--- Manual binary vs PCC_score ---")
    display(pd.crosstab(
        val_eval["manual_binary"],
        val_eval["PCC_score"],
        margins=True
    ))


# -----------------------------
# Cell 8. confusion matrix
# -----------------------------

if val is not None:
    # 3-class 자동 라벨 중 Needs review는 triage category이므로 binary 성능평가에서는 제외
    eval_binary = val_eval[val_eval["PCC_auto_label"] != "Needs review"].copy()

    eval_binary["auto_binary"] = np.where(
        eval_binary["PCC_auto_label"].eq("Likely relevant"),
        "relevant",
        "irrelevant"
    )

    labels = ["relevant", "irrelevant"]

    if len(eval_binary) > 0:
        cm = confusion_matrix(
            eval_binary["manual_binary"],
            eval_binary["auto_binary"],
            labels=labels
        )

        cm_df = pd.DataFrame(
            cm,
            index=["Manual Relevant", "Manual Irrelevant"],
            columns=["Auto Relevant", "Auto Irrelevant"]
        )

        print("\n--- Confusion matrix excluding Needs review ---")
        display(cm_df)

        print("\n--- Classification report excluding Needs review ---")
        print(classification_report(
            eval_binary["manual_binary"],
            eval_binary["auto_binary"],
            labels=labels,
            zero_division=0
        ))

        print("Cohen's kappa:", round(
            cohen_kappa_score(eval_binary["manual_binary"], eval_binary["auto_binary"]),
            3
        ))
    else:
        print("No rows available for binary evaluation after excluding Needs review.")



--- Manual binary vs PCC_auto_label ---


PCC_auto_label
manual_binary



--- Manual binary vs PCC_score ---


PCC_score
manual_binary


No rows available for binary evaluation after excluding Needs review.


In [26]:
# Cell 9. Needs review 분석
print("\n--- Needs review 문서의 자동 라벨링 플래그 분포: full corpus ---")
for col in ["P_nurse_auto", "C_speakup_voice_auto", "Ctx_healthcare_auto", "Noise_auto", "Silence_auto"]:
    print(f"\n{col}:")
    display(needs_review_df[col].value_counts(dropna=False))

print("\n--- Needs review 문서의 PCC_score 분포: full corpus ---")
display(needs_review_df["PCC_score"].value_counts(dropna=False))

if "Corpus" in needs_review_df.columns:
    print("\n--- Needs review 문서의 Corpus 분포: full corpus ---")
    display(needs_review_df["Corpus"].value_counts(dropna=False))

if val is not None:
    needs_eval = val_eval[val_eval["PCC_auto_label"] == "Needs review"].copy()
    print("\n--- Manual labels among Needs review validation rows ---")
    display(needs_eval["manual_binary"].value_counts(dropna=False))
    display(needs_eval["manual_binary"].value_counts(normalize=True, dropna=False).round(3))

    print("\n--- Needs review validation examples ---")
    cols_to_show = [c for c in ["PMID", "Title", "Abstract", manual_label_col, "manual_binary", "PCC_score", "P_nurse_auto", "C_speakup_voice_auto", "Ctx_healthcare_auto", "Noise_auto", "Silence_auto"] if c in needs_eval.columns]
    display(needs_eval[cols_to_show].head(30))



--- Needs review 문서의 자동 라벨링 플래그 분포: full corpus ---

P_nurse_auto:


,count
P_nurse_auto,
True,413
False,5



C_speakup_voice_auto:


,count
C_speakup_voice_auto,
False,418



Ctx_healthcare_auto:


...


--- Manual labels among Needs review validation rows ---


,count
manual_binary,


,proportion
manual_binary,



--- Needs review validation examples ---


,PMID,Title,Abstract,final_relevance,manual_binary,PCC_score,P_nurse_auto,C_speakup_voice_auto,Ctx_healthcare_auto,Noise_auto,Silence_auto


In [27]:
# Cell 10. Noise 분석

noise_df = df[df["Noise_auto"] == True].copy()
needs_review_noise_df = needs_review_df[needs_review_df["Noise_auto"] == True].copy()

print("\n--- Noise_auto=True 문서 수: full corpus ---")
print(len(noise_df))

if len(noise_df) > 0:
    display(noise_df["PCC_auto_label"].value_counts(dropna=False))
    display(noise_df[["PMID", "Title", "Abstract", "PCC_score", "PCC_auto_label", "Noise_auto"]].head(30))
else:
    print("No Noise_auto=True records found in full corpus.")

print("\n--- Needs review & Noise_auto=True 문서 수 ---")
print(len(needs_review_noise_df))

if len(needs_review_noise_df) > 0:
    display(needs_review_noise_df[["PMID", "Title", "Abstract", "PCC_score", "PCC_auto_label", "Noise_auto"]].head(30))
else:
    print("No Needs review records with Noise_auto=True.")

print("\n--- text_for_screening diagnostics ---")
print(df[["text_for_screening"]].info())
print("\nMissing values in text_for_screening:", df["text_for_screening"].isnull().sum())
display(df["text_for_screening"].describe())



--- Noise_auto=True 문서 수: full corpus ---
8


,count
PCC_auto_label,
Likely relevant,8


,PMID,Title,Abstract,PCC_score,PCC_auto_label,Noise_auto
494,17693833,Discharge rounds in the 80-hour workweek: impo...,BACKGROUND: Daily multidisciplinary discharge ...,3,Likely relevant,True
1261,27565996,[Briefing improves the management of a difficu...,BACKGROUND: Unanticipated airway problems in i...,3,Likely relevant,True
1879,34614129,Brazilian health professionals' perception abo...,OBJECTIVE: To describe Brazilian health profes...,2,Likely relevant,True
2144,37465068,Tracheoinnominate Artery Fistula.,AUDIENCE: This simulation provides training fo...,2,Likely relevant,True
2518,40258598,Advancing Politeness and Assertive Communicati...,BACKGROUND: Effective interprofessional commun...,3,Likely relevant,True
2532,40345655,Italian inter-societal manifesto for the preve...,"Sarcopenia, a pathology that concerns the grad...",2,Likely relevant,True
2766,41737501,The role of declining therapy volumes in skill...,INTRODUCTION: Significant declines in therapy ...,3,Likely relevant,True
2767,41756160,Perspectives on the use of artificial intellig...,INTRODUCTION: The integration of artificial in...,2,Likely relevant,True



--- Needs review & Noise_auto=True 문서 수 ---
0
No Needs review records with Noise_auto=True.

--- text_for_screening diagnostics ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3090 entries, 0 to 3089
Data columns (total 1 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   text_for_screening  3090 non-null   object
dtypes: object(1)
memory usage: 24.3+ KB
None

Missing values in text_for_screening: 0


,text_for_screening
count,3090
unique,3090
top,an unsafe environment: the reflection of a cli...
freq,1


In [28]:
# 전체 corpus
final_df = df.copy()

# validation 결과 merge
val_small = val[["PMID","manual_binary"]]

final_df["PMID"] = final_df["PMID"].astype(str)
val_small["PMID"] = val_small["PMID"].astype(str)

final_df = final_df.merge(
    val_small,
    on="PMID",
    how="left"
)

final_df["final_include"] = np.where(
    final_df["manual_binary"] == "relevant",
    True,

    np.where(
        final_df["manual_binary"] == "irrelevant",
        False,

        final_df["PCC_auto_label"] == "Likely relevant"
    )
)

final_included_df = final_df[
    final_df["final_include"] == True
].copy()

final_included_df.to_excel(
    os.path.join(
        BASE_DIR,
        "Final_Included_Corpus.xlsx"
    ),
    index=False
)

print(
    "Final Included Studies:",
    len(final_included_df)
)

/tmp/ipykernel_1872/2609789369.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_small["PMID"] = val_small["PMID"].astype(str)


Final Included Studies: 2668


텍스트 전처리 & Networking ANalysis

In [29]:
! pip install nltk

In [30]:
! pip install spacy

In [31]:
! python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 55.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [32]:
! pip install seaborn

In [33]:
! pip install plotly

In [2]:
! pip install "bertopic>=0.16.4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.1 MB/s eta 0:00:00


In [3]:
# 필요한 라이브러리 생성
import pandas as pd
import matplotlib.pyplot as plt  # ! pip install plotly (+)
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.font_manager as fm
import seaborn as sns # ! pip install seaborn & ! pip install seaborn --upgrade (+)
import numpy as np
import re
import os

# 텍스트 전처리용 라이브러리
import nltk # ! pip install nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import spacy

# TF-IDF 벡터라이저 / 키워드 추출 라이브러리
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.decomposition import LatentDirichletAllocation # LDA fallback

# BERTopic 라이브러리
from bertopic import BERTopic # ! pip install "bertopic>=0.16.4" (+)

In [4]:
# NLTK 데이터 설치 확인
# python -c " /
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')   # Python 3.10에서는 punkt_tab 사용
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [5]:
! pip install "sentence-transformers>=3.0.0"

In [6]:
! pip install "umap-learn>=0.5.7"

In [7]:
! pip install "hdbscan>=0.8.40"

In [8]:
! pip install "gensim>=4.3.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 29.4 MB/s eta 0:00:00


In [9]:
from sentence_transformers import SentenceTransformer  # ! pip install "sentence-transformers>=3.0.0"(+) 임베딩
import umap  # ! pip install "umap-learn>=0.5.7" (+) / 차원축소
import hdbscan  # ! pip install "hdbscan>=0.8.40" (+) / 클러스터링(밀도 기반)
import gensim
from gensim.models.coherencemodel import CoherenceModel # ! pip install "gensim>=4.3.3"(+) / coherence score 계산

In [10]:
! pip install wordcloud

In [11]:
! pip install pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 54.0 MB/s eta 0:00:00


In [12]:
! pip install python-louvain

In [13]:
# 워드클라우드
from wordcloud import WordCloud   # ! pip install wordcloud
import json
import csv
from pathlib import Path

# Network analysis 라이브러리
import networkx as nx
import community as community_louvain # ! pip install python-louvain / 군집화
from pyvis.network import Network # ! pip install pyvis / 네트워크 시각화

In [14]:
# 통계 라이브러리
from scipy.stats import chi2_contingency, pearsonr
from collections import Counter, defaultdict
from itertools import combinations # co-occurrence